# Ch08_03_Hydraulic

『실전! AI 프로젝트 완주의 기술』의 인쇄용 PDF에 수록된 코드에서 추출한 학습용 노트북입니다.

**주의:** 인쇄물에서 자동 추출된 초안입니다. 일부 조판 문자·줄바꿈·별도 데이터 경로가 원본 소스와 다를 수 있으므로 실행 전 확인하세요. 각 코드 셀 위에 인쇄본 페이지를 표시했습니다. 데이터 다운로드 및 패키지 설치는 필요한 경우 직접 진행해야 합니다. 출판 전 실행 검증본으로 간주하지 마세요.

**인쇄본 230쪽 — 코드** 

In [1]:
import pandas as pd
profile = pd.read_csv(
    'profile.txt',
    sep=r'\s+',
    header=None
)
profile.columns = [
    'cooler',
    'valve',
    'pump_leakage',
    'accumulator',
    'stable'
]
print(profile.shape)

(2205, 5)


**인쇄본 231쪽 — 출력 예시 (실행 코드 아님)**

```text
(2205, 5)
```

**인쇄본 231쪽 — 코드**

In [2]:
stable_idx = profile['stable'] == 0
profile_stable = profile[stable_idx].reset_index(drop=True)

**인쇄본 232쪽 — 코드**

In [3]:
print(profile_stable.shape)

(1449, 5)


**인쇄본 232쪽 — 출력 예시 (실행 코드 아님)**

```text
(1449, 5)
```

**인쇄본 232쪽 — 코드**

In [4]:
y = (profile_stable['pump_leakage'] > 0).astype(int).to_numpy()
print(pd.Series(y).value_counts().sort_index())

0    489
1    960
dtype: int64


**인쇄본 232쪽 — 출력 예시 (실행 코드 아님)**

```text
0    489
1    960
Name: count, dtype: int64
```

**인쇄본 233쪽 — 코드**

In [6]:
sensor_names = [
    'CE', 'CP', 'SE',
    'TS1', 'TS2', 'TS3', 'TS4',
    'VS1'
]
sensor_data = {}
for name in sensor_names:
    sensor_data[name] = pd.read_csv(
        name + '.txt',
        sep=r'\s+',
        header=None
    )
print(sensor_data['TS1'].shape)

(2205, 60)


**인쇄본 233쪽 — 출력 예시 (실행 코드 아님)**

```text
(2205, 60)
```

**인쇄본 233쪽 — 코드**

In [7]:
import numpy as np
features = []

**인쇄본 234쪽 — 코드**

In [8]:
for name in sensor_names:
    data = sensor_data[name].loc[stable_idx].to_numpy()
    features.append(data.mean(axis=1))
    features.append(data.std(axis=1))
X = np.column_stack(features)
print(X.shape)

(1449, 16)


**인쇄본 234쪽 — 출력 예시 (실행 코드 아님)**

```text
(1449, 16)
```

**인쇄본 234쪽 — 코드**

In [9]:
from sklearn.model_selection import train_test_split
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.4,
    stratify=y,
    random_state=0
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    stratify=y_temp,
    random_state=0
)
print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

(869, 16)
(290, 16)
(290, 16)


**인쇄본 235쪽 — 출력 예시 (실행 코드 아님)**

```text
(869, 16)
(290, 16)
(290, 16)
```

**인쇄본 235쪽 — 코드**

In [10]:
X_train_normal = X_train[y_train == 0]
print(X_train_normal.shape)

(293, 16)


**인쇄본 235쪽 — 출력 예시 (실행 코드 아님)**

```text
(293, 16)
```

**인쇄본 236쪽 — 코드**

In [11]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_normal = scaler.fit_transform(
    X_train_normal
)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

**인쇄본 237쪽 — 코드**

In [12]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense
model = Sequential([
    Input(shape=(16,)),
    Dense(8, activation='relu'),
    Dense(3, activation='relu'),
    Dense(8, activation='relu'),
    Dense(16)
])
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 8)                 136       
                                                                 
 dense_1 (Dense)             (None, 3)                 27        
                                                                 
 dense_2 (Dense)             (None, 8)                 32        
                                                                 
 dense_3 (Dense)             (None, 16)                144       
                                                                 
Total params: 339
Trainable params: 339
Non-trainable params: 0
_________________________________________________________________


**인쇄본 238쪽 — 코드**

In [13]:
model.compile(
    optimizer='adam',
    loss='mse'
)
history = model.fit(
    X_train_normal,
    X_train_normal,
    epochs=50,
    batch_size=32,
    verbose=0
)

**인쇄본 239쪽 — 코드**

In [14]:
import numpy as np
X_val_hat = model.predict(X_val,verbose=0)
val_error = np.sqrt(
    np.mean(np.square(X_val - X_val_hat),axis=1)
)

**인쇄본 239쪽 — 코드**

In [15]:
val_error_normal = val_error[y_val == 0]
val_error_leakage = val_error[y_val == 1]
print("Normal =", val_error_normal.mean())
print("Leakage =", val_error_leakage.mean())

Normal = 0.504701294015816
Leakage = 0.9070425948274686


**인쇄본 239쪽 — 출력 예시 (실행 코드 아님)**

```text
Normal = 0.287
Leakage = 0.521
```

**인쇄본 240쪽 — 코드**

In [16]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)
tau_candidates = np.linspace(
    val_error.min(),
    val_error.max(),
    200
)
best_tau = None
best_f1 = -1
for tau in tau_candidates:
    y_val_pred = (val_error > tau).astype(int)
    precision = precision_score(
        y_val, y_val_pred,
        zero_division=0
    )
    recall = recall_score(
        y_val, y_val_pred,
        zero_division=0
    )
    f1 = f1_score(
        y_val, y_val_pred,
        zero_division=0
    )
    
    if f1 > best_f1:
        best_f1 = f1
        best_tau = tau
        
print("tau =", best_tau)
print("F1 score =", best_f1)

tau = 0.33375844347154315
F1 score = 0.8532110091743119


**인쇄본 241쪽 — 출력 예시 (실행 코드 아님)**

```text
tau = 0.4345
F1 score = 0.8558
```

**인쇄본 241쪽 — 코드**

In [17]:
X_test_hat = model.predict(X_test,verbose=0)
test_error = np.sqrt(
    np.mean(np.square(X_test - X_test_hat),axis=1)
)
y_test_pred = (test_error > best_tau).astype(int)

**인쇄본 242쪽 — 코드**

In [18]:
from sklearn.metrics import confusion_matrix
precision = precision_score(y_test, y_test_pred)
recall = recall_score(y_test, y_test_pred)
f1 = f1_score(y_test, y_test_pred)
cm = confusion_matrix(y_test, y_test_pred)
print("Precision =", precision)
print("Recall =", recall)
print("F1 score =", f1)
print(cm)

Precision = 0.752
Recall = 0.9791666666666666
F1 score = 0.8506787330316741
[[ 36  62]
 [  4 188]]


**인쇄본 242쪽 — 출력 예시 (실행 코드 아님)**

```text
Precision = 0.7824
Recall = 0.9740
F1 score = 0.8677
[[ 46  52]
 [  5 187]]
```